# 03 — Property Prediction & Drug-Likeness

**No API key required.** Predict molecular properties, rank candidates, and assess drug-likeness.

Capabilities:
- RDKit property prediction (MW, LogP, TPSA, QED, SA score)
- Drug score computation
- Synthesisability classification
- Candidate ranking by QED

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import matplotlib.pyplot as plt
from chemvision.generation.property_predictor import PropertyPredictor

pred = PropertyPredictor(use_mace=False)
print('PropertyPredictor ready (RDKit backend)')

## 3.1 — Predict properties for a single molecule

In [ ]:
result = pred.predict('CC(=O)Oc1ccccc1C(=O)O')  # aspirin

print(f'SMILES:           {result.smiles}')
print(f'MW:               {result.mw:.1f} g/mol')
print(f'LogP:             {result.logp:.2f}')
print(f'TPSA:             {result.tpsa:.1f} A^2')
print(f'QED:              {result.qed:.3f}')
print(f'SA score:         {result.sa_score:.1f} (1=easy, 10=hard)')
print(f'Drug score:       {result.drug_score:.3f}')
print(f'Synthesisability: {result.synthesisability}')
print(f'HBD / HBA:        {result.hbd} / {result.hba}')
print(f'Backend:          {result.backend}')

## 3.2 — Rank a set of drug candidates

In [ ]:
candidates = [
    ('Aspirin', 'CC(=O)Oc1ccccc1C(=O)O'),
    ('Ibuprofen', 'CC(C)Cc1ccc(cc1)C(C)C(=O)O'),
    ('Caffeine', 'Cn1cnc2c1c(=O)n(c(=O)n2C)C'),
    ('Paracetamol', 'CC(=O)Nc1ccc(O)cc1'),
    ('Metformin', 'CN(C)C(=N)NC(=N)N'),
    ('Penicillin V', 'CC1(C(N2C(S1)C(C2=O)NC(=O)COc3ccccc3)C(=O)O)C'),
]

ranked = pred.rank_candidates([smi for _, smi in candidates])

rows = []
name_map = {smi: name for name, smi in candidates}
for i, r in enumerate(ranked):
    rows.append({
        'Rank': i + 1,
        'Name': name_map.get(r.smiles, r.smiles[:30]),
        'QED': f'{r.qed:.3f}' if r.qed else '—',
        'MW': f'{r.mw:.1f}' if r.mw else '—',
        'LogP': f'{r.logp:.2f}' if r.logp is not None else '—',
        'SA': f'{r.sa_score:.1f}' if r.sa_score else '—',
        'Drug score': f'{r.drug_score:.3f}' if r.drug_score else '—',
        'Synth.': r.synthesisability,
    })

pd.DataFrame(rows)

In [ ]:
# Scatter: QED vs MW
fig, ax = plt.subplots(figsize=(8, 5))
for r in ranked:
    name = name_map.get(r.smiles, '?')
    ax.scatter(r.mw, r.qed, s=100, zorder=3)
    ax.annotate(name, (r.mw, r.qed), textcoords='offset points',
                xytext=(8, 5), fontsize=9)
ax.set_xlabel('Molecular Weight (g/mol)')
ax.set_ylabel('QED (drug-likeness)')
ax.set_title('Drug candidate landscape')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()